# Semana 14: Integrales y Aplicaciones en Evaluación de Modelos
## Notebook Conceptual (NB1) - Explorando Integrales y AUC

### Propósito de la sesión
Conectar la **integral definida** con métricas clave en machine learning, especialmente el **Área Bajo la Curva (AUC)**. Exploraremos el concepto de integral como área bajo una curva, calcularemos integrales numéricamente con `scipy.integrate.quad`, y aplicaremos estos conceptos para calcular el AUC de un clasificador dummy a partir de su curva ROC.

### Objetivos de aprendizaje
*   Comprender la integral definida como el área bajo una curva.
*   Calcular integrales definidas numéricamente usando `scipy.integrate.quad`.
*   Visualizar el área bajo curvas de funciones simples.
*   Generar predicciones de un clasificador dummy y dibujar la **curva ROC**.
*   Calcular el **AUC** (Área Bajo la Curva ROC) mediante integración numérica.
*   Conectar estos conceptos con aplicaciones en ML (AUC), DL (entropía cruzada), GenAI (ELBO en VAEs) y probabilidad.

## Configuración Inicial

Importamos las librerías necesarias: numpy, matplotlib, scipy para integración y sklearn para métricas.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
from sklearn import metrics

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Idea Intuitiva de Integral: Área Bajo una Curva

La integral definida $\int_a^b f(x) \, dx$ representa el área neta bajo la curva $f(x)$ entre $x=a$ y $x=b$ (considerando áreas negativas cuando $f(x) < 0$).

In [ ]:
# Función a integrar: f(x) = x^2 entre 0 y 1
def f_cuad(x):
    return x**2

a, b = 0, 1

# Visualización
x_vals = np.linspace(a-0.2, b+0.2, 400)
y_vals = f_cuad(x_vals)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_vals, y_vals, 'b-', linewidth=2)
plt.fill_between(x_vals, y_vals, where=(x_vals >= a) & (x_vals <= b), color='skyblue', alpha=0.5)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(a, color='red', linestyle='--', label=f'x={a}')
plt.axvline(b, color='red', linestyle='--', label=f'x={b}')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title(r'$\int_0^1 x^2 \, dx$ (área sombreada)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
# Aproximación con rectángulos (suma de Riemann)
n_rect = 10
x_rect = np.linspace(a, b, n_rect+1)
width = (b-a)/n_rect
for i in range(n_rect):
    plt.bar(x_rect[i], f_cuad(x_rect[i]), width=width, align='edge', alpha=0.5, color='green', edgecolor='black')
plt.plot(x_vals, y_vals, 'b-', linewidth=2)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title(f'Aproximación con {n_rect} rectángulos')
plt.grid(True)

plt.suptitle('Interpretación geométrica de la integral definida', fontsize=14)
plt.tight_layout()
plt.show()

---
## Actividad 1: Calcular integrales definidas con `scipy.integrate.quad`

La función `quad` de scipy.integrate calcula numéricamente la integral definida con alta precisión.

In [ ]:
# Integral de x^2 de 0 a 1
resultado, error = integrate.quad(f_cuad, 0, 1)
print(f"∫₀¹ x² dx = {resultado:.6f} (exacto: 1/3 = {1/3:.6f})")
print(f"Error estimado: {error:.2e}")

# Integral de sin(x) de 0 a π
res_sin, _ = integrate.quad(np.sin, 0, np.pi)
print(f"\n∫₀^π sin(x) dx = {res_sin:.6f} (exacto: 2)")

# Integral de exp(-x) de 0 a ∞
res_exp, _ = integrate.quad(lambda x: np.exp(-x), 0, np.inf)
print(f"∫₀^∞ e⁻ˣ dx = {res_exp:.6f} (exacto: 1)")

# Integral de 1/(1+x²) de -∞ a ∞ (área bajo la campana de Cauchy)
res_cauchy, _ = integrate.quad(lambda x: 1/(1+x**2), -np.inf, np.inf)
print(f"∫_{-∞}^∞ 1/(1+x²) dx = {res_cauchy:.6f} (exacto: π = {np.pi:.6f})")

---
## 2. Aplicación en Probabilidad: Funciones de Densidad

La integral de una función de densidad de probabilidad (PDF) sobre un intervalo da la probabilidad de que la variable aleatoria caiga en ese intervalo.

In [ ]:
# PDF de la distribución normal estándar N(0,1)
def pdf_normal(x):
    return (1/np.sqrt(2*np.pi)) * np.exp(-x**2 / 2)

# Visualizamos la PDF
x_norm = np.linspace(-4, 4, 400)
y_norm = pdf_normal(x_norm)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_norm, y_norm, 'b-', linewidth=2)
plt.fill_between(x_norm, y_norm, where=(x_norm >= -1) & (x_norm <= 1), color='skyblue', alpha=0.5)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('PDF Normal Estándar')
plt.grid(True)

plt.subplot(1, 2, 2)
# Probabilidad P(-1 ≤ X ≤ 1)
prob, _ = integrate.quad(pdf_normal, -1, 1)
print(f"P(-1 ≤ X ≤ 1) = {prob:.4f}")

# Comparamos con la CDF de scipy
from scipy.stats import norm
prob_cdf = norm.cdf(1) - norm.cdf(-1)
print(f"Usando CDF: {prob_cdf:.4f}")

# Visualizamos el área como barra
plt.bar(['P(-1 ≤ X ≤ 1)'], [prob], color='skyblue', alpha=0.7)
plt.axhline(y=0.6827, color='r', linestyle='--', label='68.27% (teórico)')
plt.ylabel('Probabilidad')
plt.title('Probabilidad en el intervalo [-1,1]')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

---
## Actividad 2: Curva ROC y AUC (Área Bajo la Curva)

La **curva ROC** (Receiver Operating Characteristic) representa la tasa de verdaderos positivos (TPR) frente a la tasa de falsos positivos (FPR) al variar el umbral de decisión de un clasificador binario.

El **AUC** (Area Under the Curve) es la integral de TPR respecto a FPR:

$$\text{AUC} = \int_0^1 \text{TPR}(\text{FPR}) \, d(\text{FPR})$$

### 2.1. Generación de datos de un clasificador dummy

In [ ]:
# Generamos etiquetas verdaderas (50 positivos, 50 negativos)
y_true = np.array([0]*50 + [1]*50)

# Simulamos puntuaciones de un clasificador:
# Los positivos tienden a tener puntuaciones altas, los negativos bajas
np.random.seed(42)
scores_pos = np.random.normal(0.8, 0.2, 50)   # positivos: media alta
scores_neg = np.random.normal(0.3, 0.2, 50)   # negativos: media baja
y_score = np.concatenate([scores_neg, scores_pos])

# Visualizamos distribuciones
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(scores_neg, bins=15, alpha=0.7, label='Negativos', color='red', edgecolor='black')
plt.hist(scores_pos, bins=15, alpha=0.7, label='Positivos', color='blue', edgecolor='black')
plt.xlabel('Puntuación')
plt.ylabel('Frecuencia')
plt.title('Distribución de puntuaciones por clase')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
# Mostramos las primeras puntuaciones
plt.scatter(range(len(y_score)), y_score, c=y_true, cmap='bwr', alpha=0.6, edgecolors='k')
plt.xlabel('Índice de muestra')
plt.ylabel('Puntuación')
plt.title('Puntuaciones coloreadas por clase (rojo=negativo, azul=positivo)')
plt.grid(True)
plt.tight_layout()
plt.show()

### 2.2. Cálculo de la curva ROC y AUC

In [ ]:
# Calculamos TPR y FPR para diferentes umbrales
fpr, tpr, thresholds = metrics.roc_curve(y_true, y_score)

# Calculamos AUC con dos métodos
auc_trapz = np.trapz(tpr, fpr)  # integración trapezoidal
auc_sklearn = metrics.roc_auc_score(y_true, y_score)

print(f"AUC calculado con trapz: {auc_trapz:.4f}")
print(f"AUC con sklearn: {auc_sklearn:.4f}")

# Visualizamos la curva ROC
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc_sklearn:.4f})')
plt.fill_between(fpr, tpr, alpha=0.3, color='skyblue')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Clasificador aleatorio (AUC=0.5)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

### 2.3. Explicación del AUC como integral

El AUC es el área bajo la curva ROC, que se calcula como la integral de TPR con respecto a FPR. Cuanto más se acerca a 1, mejor es el clasificador. Un valor de 0.5 indica un clasificador aleatorio.

In [ ]:
# Demostración de cómo se construye la curva variando el umbral
umbrales_ejemplo = [0.2, 0.5, 0.8]

plt.figure(figsize=(14, 8))

for i, umbral in enumerate(umbrales_ejemplo):
    y_pred = (y_score >= umbral).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    
    tpr_umbral = tp / (tp + fn) if (tp+fn)>0 else 0
    fpr_umbral = fp / (fp + tn) if (fp+tn)>0 else 0
    
    plt.subplot(2, 3, i+1)
    plt.scatter(range(len(y_score)), y_score, c=y_true, cmap='bwr', alpha=0.6)
    plt.axhline(y=umbral, color='green', linestyle='--', label=f'Umbral={umbral}')
    plt.xlabel('Muestra')
    plt.ylabel('Puntuación')
    plt.title(f'Umbral = {umbral}')
    plt.legend()
    
    plt.subplot(2, 3, i+4)
    plt.plot(fpr, tpr, 'b-', alpha=0.5)
    plt.plot(fpr_umbral, tpr_umbral, 'ro', markersize=10)
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title(f'Punto en ROC: (FPR={fpr_umbral:.2f}, TPR={tpr_umbral:.2f})')
    plt.grid(True)

plt.suptitle('Efecto del umbral en la clasificación y en la curva ROC', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Conexiones IA

### 3.1. ML: AUC como métrica de evaluación
El AUC es una métrica fundamental para clasificadores binarios, independiente del umbral. Se interpreta como la probabilidad de que el modelo asigne una puntuación más alta a un positivo aleatorio que a un negativo aleatorio.

### 3.2. DL: Entropía cruzada
La entropía cruzada tiene origen en teoría de la información y se expresa como una integral (o suma) de logaritmos. Para distribuciones continuas:

$$H(p,q) = -\int p(x) \log q(x) \, dx$$

### 3.3. GenAI: ELBO en VAEs
En Autoencoders Variacionales, se maximiza una cota inferior de la verosimilitud (ELBO) que contiene una esperanza (integral sobre la distribución latente).

### 3.4. Probabilidad: Cálculo de probabilidades
La integral de la PDF da probabilidades acumuladas.

In [ ]:
# Ejemplo: Entropía cruzada para distribuciones continuas (simulación)
# p(x) distribución real, q(x) distribución estimada
def p(x):
    return pdf_normal(x)  # N(0,1)

def q(x):
    # Estimación: normal con media 0.5 y desviación 1.2
    return (1/(np.sqrt(2*np.pi)*1.2)) * np.exp(-(x-0.5)**2 / (2*1.2**2))

# Calculamos la entropía cruzada H(p,q) = -∫ p(x) log q(x) dx
def integrando_cross(x):
    return -p(x) * np.log(q(x) + 1e-10)

H_pq, _ = integrate.quad(integrando_cross, -np.inf, np.inf)
print(f"Entropía cruzada H(p,q) ≈ {H_pq:.4f} nats")

# Visualizamos las distribuciones
x_vals = np.linspace(-5, 5, 400)
plt.figure(figsize=(10, 6))
plt.plot(x_vals, p(x_vals), 'b-', label='p(x) (real)')
plt.plot(x_vals, q(x_vals), 'r--', label='q(x) (estimada)')
plt.fill_between(x_vals, p(x_vals), alpha=0.3, color='blue')
plt.xlabel('x')
plt.ylabel('Densidad')
plt.title('Distribuciones real y estimada')
plt.legend()
plt.grid(True)
plt.show()

---
## Ejercicios para el Estudiante

1.  **Integral de una función a trozos:** Define una función a trozos y calcula su integral en un intervalo usando `quad`. Por ejemplo, $f(x) = x$ para $x<0$, $f(x)=x^2$ para $x\geq 0$, de -1 a 2.

2.  **AUC para clasificador perfecto y aleatorio:** Genera puntuaciones para un clasificador perfecto (positivos siempre > negativos) y para uno aleatorio (distribuciones iguales). Calcula sus AUC y dibuja las curvas ROC.

3.  **Comparación de clasificadores:** Genera dos clasificadores simulados (uno mejor que otro) y compara sus curvas ROC en la misma gráfica. ¿Cómo se refleja en el AUC?

4.  **Probabilidad de una normal:** Calcula la probabilidad de que una variable normal estándar esté entre -2 y 2. Compara con el valor teórico (aprox. 0.9545).

5.  **Reflexión:** ¿Por qué el AUC es independiente del umbral? ¿Qué ventajas tiene frente a la precisión (accuracy) para conjuntos desbalanceados?

---
## Conclusiones de la Sesión

*   La **integral definida** mide el área bajo una curva y es fundamental en probabilidad y evaluación de modelos.
*   Hemos calculado integrales numéricamente con `scipy.integrate.quad` y visualizado el área bajo curvas de funciones simples.
*   Generamos predicciones de un clasificador dummy y construimos su **curva ROC**, visualizando cómo varía TPR y FPR con el umbral.
*   Calculamos el **AUC** mediante integración trapezoidal, verificando con `sklearn.metrics.roc_auc_score`.
*   Conectamos estos conceptos con aplicaciones en:
    *   **ML:** AUC como métrica de evaluación de clasificadores binarios.
    *   **DL:** Entropía cruzada como integral.
    *   **GenAI:** ELBO en VAEs (expectativas como integrales).
    *   **Probabilidad:** Cálculo de probabilidades mediante integración de PDFs.

¡Con esta sesión cerramos el curso, integrando el cálculo y el álgebra lineal con aplicaciones prácticas en inteligencia artificial!